# Ultrasound Image Preprocessing Pipeline

This notebook handles:
1. Image resizing and normalization
2. Perceptual hashing for duplicate detection
3. Group-level train/val/test splitting
4. Saving processed images organized by split and group

## Mount Google Drive

In [24]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted


## Import Libraries

In [25]:
!pip install imagehash

In [26]:
import os
import json
import random
import pickle
import hashlib
import multiprocessing as mp
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import imagehash

import torch


print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
  print(f"Using device: {device}")

GPU: Tesla T4
Using device: cuda


## Configuration

In [27]:
DRIVE_BASE = Path('/content/drive/MyDrive/Data/liver')
RAW_DATA_DIR = DRIVE_BASE / 'Dataset' / 'Dataset'
# OUTPUT_DIR = DRIVE_BASE / 'preprocessed'
OUTPUT_DIR = Path('/content/drive/MyDrive/local/preprocessed')


TARGET_SIZE = (224, 224)
CLASSES = ['F0', 'F1', 'F2', 'F3', 'F4']

CONFIG = {
    'target_size': TARGET_SIZE,
    'hash_size': 128,
    'similarity_threshold': 10,
    'val_split': 0.15,
    'test_split': 0.15,
    'random_seed': 42,
}

random.seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

print(f"Raw data: {RAW_DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Target size: {TARGET_SIZE}")

Raw data: /content/drive/MyDrive/Data/liver/Dataset/Dataset
Output: /content/drive/MyDrive/local/preprocessed
Target size: (224, 224)


## Explore Raw Data

In [28]:
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data directory not found: {RAW_DATA_DIR}")

all_classes = sorted([d.name for d in RAW_DATA_DIR.iterdir() if d.is_dir()])
print(f"Found classes: {all_classes}")

all_images = []
for cls_idx, cls in enumerate(CLASSES):
    cls_path = RAW_DATA_DIR / cls
    if not cls_path.is_dir():
        print(f"Warning: {cls} directory not found")
        continue

    for img_file in cls_path.iterdir():
        if img_file.is_file():
            all_images.append((str(img_file), cls_idx, cls))

print("\nClass distribution:")
for i, cls in enumerate(CLASSES):
    count = sum(1 for _, label, _ in all_images if label == i)
    print(f"  {cls}: {count} images")

print(f"Total images: {len(all_images)}")

Found classes: ['F0', 'F1', 'F2', 'F3', 'F4']

Class distribution:
  F0: 2114 images
  F1: 861 images
  F2: 793 images
  F3: 857 images
  F4: 1698 images
Total images: 6323


In [29]:
print("Analyzing image dimensions...\n")

for cls in all_classes:
    cls_path = RAW_DATA_DIR / cls
    shapes = []

    for img_file in cls_path.iterdir():
        if not img_file.is_file():
            continue
        try:
            with Image.open(img_file) as im:
                shapes.append(im.size)
        except Exception as e:
            print(f"  Warning: Could not read {img_file.name}: {e}")

    dim_counts = Counter(shapes)
    top_dims = dim_counts.most_common(5)

    print(f"Class {cls}:")
    for (w, h), count in top_dims:
        print(f"  {w}x{h}: {count} images")
    print()

Analyzing image dimensions...

Class F0:
  640x480: 2114 images

Class F1:
  449x464: 489 images
  640x480: 372 images

Class F2:
  449x464: 793 images

Class F3:
  449x464: 857 images

Class F4:
  640x480: 973 images
  449x464: 725 images



## Image Processing Functions

In [30]:
def process_single_image(input_path, output_path, target_size=(224, 224)):
    try:
        with Image.open(input_path) as img:
            original_size = img.size

            if img.mode not in ['L', 'RGB']:
                img = img.convert('RGB')

            width, height = img.size
            target_w, target_h = target_size

            scale = min(target_w / width, target_h / height)
            new_width = int(width * scale)
            new_height = int(height * scale)

            resized = img.resize((new_width, new_height), Image.Resampling.LANCZOS)

            if img.mode == 'L':
                padded = Image.new('L', target_size, 0)
            else:
                padded = Image.new('RGB', target_size, (0, 0, 0))

            paste_x = (target_w - new_width) // 2
            paste_y = (target_h - new_height) // 2
            padded.paste(resized, (paste_x, paste_y))

            arr = np.array(padded, dtype=np.float32)

            if arr.ndim == 2:
                min_val, max_val = arr.min(), arr.max()
                if max_val > min_val:
                    arr = (arr - min_val) / (max_val - min_val) * 255.0
            else:
                for c in range(arr.shape[2]):
                    channel = arr[:, :, c]
                    min_val, max_val = channel.min(), channel.max()
                    if max_val > min_val:
                        arr[:, :, c] = (channel - min_val) / (max_val - min_val) * 255.0

            arr = np.round(arr).astype(np.uint8)
            processed = Image.fromarray(arr, mode=padded.mode)
            processed.save(output_path, 'PNG')

            return {'success': True, 'original_size': original_size}

    except Exception as e:
        return {'success': False, 'error': str(e)}

print("Processing function defined")

Processing function defined


## Perceptual Hashing Functions

In [31]:
def compute_perceptual_hash(img_path, hash_size=16):
    try:
        img = Image.open(img_path)
        h = imagehash.phash(img, hash_size=hash_size)
        return h.hash.flatten().astype(np.uint8)
    except Exception:
        return None

def compute_hash_batch(args):
    filepath, label, hash_size = args
    img_hash = compute_perceptual_hash(filepath, hash_size)
    return filepath, img_hash, label

print("Hashing functions defined (using imagehash phash)")

Hashing functions defined (using imagehash phash)


## Compute Perceptual Hashes

In [32]:
print("="*60)
print("COMPUTING PERCEPTUAL HASHES")
print("="*60)

hash_args = [(filepath, label, CONFIG['hash_size'])
             for filepath, label, _ in all_images]

num_workers = mp.cpu_count()
print(f"Using {num_workers} CPU cores")

image_hashes = {}

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(compute_hash_batch, arg): arg for arg in hash_args}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Hashing"):
        filepath, img_hash, label = future.result()
        if img_hash is not None:
            image_hashes[filepath] = (img_hash, label)

print(f"Hashed {len(image_hashes)} images")

COMPUTING PERCEPTUAL HASHES
Using 2 CPU cores


Hashing: 100%|██████████| 6323/6323 [01:27<00:00, 72.05it/s]

Hashed 6323 images


## Find Similar Image Groups

In [33]:
print("\n" + "="*60)
print("FINDING SIMILAR IMAGE GROUPS (GPU-accelerated)")
print("="*60)

all_paths = list(image_hashes.keys())

hash_list = [image_hashes[path][0] for path in all_paths]
hash_matrix = torch.tensor(np.array(hash_list), dtype=torch.uint8, device=device)
n_images = len(all_paths)

print(f"Total images: {n_images}")
print(f"Hash size: {hash_matrix.shape[1]} bits")
print(f"Using: {device}")

parent = list(range(n_images))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py


FINDING SIMILAR IMAGE GROUPS (GPU-accelerated)
Total images: 6323
Hash size: 16384 bits
Using: cuda


In [34]:
batch_size = 2000
threshold = CONFIG['similarity_threshold']
all_matches = []

for i in tqdm(range(n_images), desc="Finding groups (GPU)"):
    if i + 1 >= n_images:
        continue

    hash_i = hash_matrix[i:i+1]
    remaining = hash_matrix[i+1:]

    for start in range(0, remaining.shape[0], batch_size):
        end = min(start + batch_size, remaining.shape[0])
        batch = remaining[start:end]

        distances = torch.sum(hash_i != batch, dim=1)
        matches_idx = torch.where(distances <= threshold)[0]

        for j in matches_idx.cpu().tolist():
            all_matches.append((i, i + 1 + start + j))

print(f"Found {len(all_matches)} similar pairs")

for i, j in all_matches:
    union(i, j)

groups_dict = defaultdict(list)
for i, path in enumerate(all_paths):
    root = find(i)
    groups_dict[root].append(path)

similar_groups = [group for group in groups_dict.values() if len(group) > 1]

n_in_groups = sum(len(g) for g in similar_groups)
n_singletons = n_images - n_in_groups

print("\n" + "="*60)
print("DUPLICATE DETECTION RESULTS")
print("="*60)
print(f"Total images: {len(all_paths)}")
print(f"Images in similar groups: {n_in_groups}")
print(f"Unique singleton images: {n_singletons}")
print(f"Number of similar groups: {len(similar_groups)}")
print(f"Total unique sources: {len(similar_groups) + n_singletons}")
print("="*60)

Finding groups (GPU): 100%|██████████| 6323/6323 [00:32<00:00, 196.17it/s] 

Found 14047 similar pairs

DUPLICATE DETECTION RESULTS
Total images: 6323
Images in similar groups: 6225
Unique singleton images: 98
Number of similar groups: 1438
Total unique sources: 1536


## Assign Group IDs

In [35]:
path_to_group = {}
group_id = 0

for group in similar_groups:
    for path in group:
        path_to_group[path] = group_id
    group_id += 1

for path in all_paths:
    if path not in path_to_group:
        path_to_group[path] = group_id
        group_id += 1

print(f"Total unique groups: {group_id}")

Total unique groups: 1536


## Train/Val/Test Split at Group Level

In [36]:
group_to_paths = defaultdict(list)
group_to_label = {}

for path in all_paths:
    gid = path_to_group[path]
    group_to_paths[gid].append(path)
    group_to_label[gid] = image_hashes[path][1]

all_group_ids = list(group_to_paths.keys())
all_group_labels = [group_to_label[gid] for gid in all_group_ids]

print(f"Splitting {len(all_group_ids)} groups")

print("\nClass distribution (groups):")
for i, cls_name in enumerate(CLASSES):
    count = sum(1 for lbl in all_group_labels if lbl == i)
    print(f"  {cls_name}: {count} groups")

Splitting 1536 groups

Class distribution (groups):
  F0: 317 groups
  F1: 296 groups
  F2: 308 groups
  F3: 308 groups
  F4: 307 groups


In [37]:
train_group_idx, temp_test_group_idx = train_test_split(
    range(len(all_group_ids)),
    test_size=(CONFIG['val_split'] + CONFIG['test_split']),
    stratify=all_group_labels,
    random_state=CONFIG['random_seed']
)

temp_test_labels = [all_group_labels[i] for i in temp_test_group_idx]
val_group_idx_local, test_group_idx_local = train_test_split(
    range(len(temp_test_group_idx)),
    test_size=0.5,
    stratify=temp_test_labels,
    random_state=CONFIG['random_seed']
)

val_group_idx = [temp_test_group_idx[i] for i in val_group_idx_local]
test_group_idx = [temp_test_group_idx[i] for i in test_group_idx_local]

train_groups = set(all_group_ids[i] for i in train_group_idx)
val_groups = set(all_group_ids[i] for i in val_group_idx)
test_groups = set(all_group_ids[i] for i in test_group_idx)

train_paths = []
val_paths = []
test_paths = []

for gid, paths in group_to_paths.items():
    if gid in train_groups:
        train_paths.extend([(p, group_to_label[gid], gid) for p in paths])
    elif gid in val_groups:
        val_paths.extend([(p, group_to_label[gid], gid) for p in paths])
    else:
        test_paths.extend([(p, group_to_label[gid], gid) for p in paths])

print("\n" + "="*60)
print("TRAIN/VAL/TEST SPLIT")
print("="*60)
print(f"Train: {len(train_groups)} groups, {len(train_paths)} images ({len(train_paths)/len(all_paths)*100:.1f}%)")
print(f"Val:   {len(val_groups)} groups, {len(val_paths)} images ({len(val_paths)/len(all_paths)*100:.1f}%)")
print(f"Test:  {len(test_groups)} groups, {len(test_paths)} images ({len(test_paths)/len(all_paths)*100:.1f}%)")
print("="*60)


TRAIN/VAL/TEST SPLIT
Train: 1075 groups, 4402 images (69.6%)
Val:   230 groups, 963 images (15.2%)
Test:  231 groups, 958 images (15.2%)


In [38]:
overlap_train_val = train_groups & val_groups
overlap_train_test = train_groups & test_groups
overlap_val_test = val_groups & test_groups

if overlap_train_val or overlap_train_test or overlap_val_test:
    raise ValueError("Data leakage detected!")
else:
    print("Verified: No group overlap between train/val/test")
    print("Data leakage prevented")

Verified: No group overlap between train/val/test
Data leakage prevented


## Process and Save Images

In [39]:
for split in ['train', 'val', 'test']:
    for cls in CLASSES:
        (OUTPUT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print(f"Created output directories at: {OUTPUT_DIR}")

Created output directories at: /content/drive/MyDrive/local/preprocessed


In [40]:
def process_split(split_name, paths_data):
    stats = {'total': 0, 'success': 0, 'failed': 0}
    processed_info = []

    print(f"\nProcessing {split_name}: {len(paths_data)} images")

    for src_path, label, group_id in tqdm(paths_data, desc=f"  {split_name}"):
        cls_name = CLASSES[label]
        src_file = Path(src_path)
        output_fname = f"g{group_id:05d}_{src_file.stem}.png"
        output_path = OUTPUT_DIR / split_name / cls_name / output_fname

        result = process_single_image(src_path, output_path, TARGET_SIZE)

        stats['total'] += 1
        if result['success']:
            stats['success'] += 1
            processed_info.append({
                'src': src_path,
                'dst': str(output_path),
                'class': cls_name,
                'label': label,
                'group_id': group_id,
            })
        else:
            stats['failed'] += 1

    return stats, processed_info

train_stats, train_info = process_split('train', train_paths)
val_stats, val_info = process_split('val', val_paths)
test_stats, test_info = process_split('test', test_paths)

print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Train - Total: {train_stats['total']}, Success: {train_stats['success']}, Failed: {train_stats['failed']}")
print(f"Val   - Total: {val_stats['total']}, Success: {val_stats['success']}, Failed: {val_stats['failed']}")
print(f"Test  - Total: {test_stats['total']}, Success: {test_stats['success']}, Failed: {test_stats['failed']}")
print("="*60)


Processing train: 4402 images


  train:   0%|          | 0/4402 [00:00<?, ?it/s]/tmp/ipython-input-801358089.py:41: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  processed = Image.fromarray(arr, mode=padded.mode)
  train: 100%|██████████| 4402/4402 [01:39<00:00, 44.40it/s]



Processing val: 963 images


  val: 100%|██████████| 963/963 [00:22<00:00, 43.42it/s]



Processing test: 958 images


  test: 100%|██████████| 958/958 [00:21<00:00, 44.35it/s]


PROCESSING COMPLETE
Train - Total: 4402, Success: 4402, Failed: 0
Val   - Total: 963, Success: 963, Failed: 0
Test  - Total: 958, Success: 958, Failed: 0


## Verify Processed Data

In [41]:
print("Verifying processed images...\n")

for split in ['train', 'val', 'test']:
    print(f"{split.upper()}:")
    for cls in CLASSES:
        cls_path = OUTPUT_DIR / split / cls
        if cls_path.is_dir():
            count = len(list(cls_path.glob('*.png')))
            print(f"  {cls}: {count} images")
    print()

Verifying processed images...

TRAIN:
  F0: 1590 images
  F1: 724 images
  F2: 652 images
  F3: 722 images
  F4: 1335 images

VAL:
  F0: 331 images
  F1: 174 images
  F2: 140 images
  F3: 159 images
  F4: 297 images

TEST:
  F0: 312 images
  F1: 157 images
  F2: 144 images
  F3: 176 images
  F4: 321 images



## Save Metadata

In [42]:
metadata = {
    'version': '3.0',
    'config': CONFIG,
    'classes': CLASSES,
    'num_classes': len(CLASSES),
    'statistics': {
        'train': {'images': len(train_paths), 'groups': len(train_groups)},
        'val': {'images': len(val_paths), 'groups': len(val_groups)},
        'test': {'images': len(test_paths), 'groups': len(test_groups)},
        'similar_groups_found': len(similar_groups),
        'total_images': len(all_paths),
        'total_groups': group_id,
    },
    'train_files': train_info,
    'val_files': val_info,
    'test_files': test_info,
}

metadata_path = OUTPUT_DIR / 'preprocessing_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")

Metadata saved to: /content/drive/MyDrive/local/preprocessed/preprocessing_metadata.json


In [43]:
summary_metadata = {
    'version': '3.0',
    'config': CONFIG,
    'classes': CLASSES,
    'num_classes': len(CLASSES),
    'statistics': metadata['statistics'],
}

summary_path = OUTPUT_DIR / 'metadata_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary_metadata, f, indent=2)

print(f"Summary saved to: {summary_path}")
print("\nPreprocessing complete")

Summary saved to: /content/drive/MyDrive/local/preprocessed/metadata_summary.json

Preprocessing complete


## Summary

Output structure:
```
preprocessed/
    train/
        F0/, F1/, F2/, F3/, F4/
    val/
        F0/, F1/, F2/, F3/, F4/
    test/
        F0/, F1/, F2/, F3/, F4/
    preprocessing_metadata.json
    metadata_summary.json
```

Files are named as: `g{group_id}_{original_name}.png`

Groups are kept together in the same split to prevent data leakage.